## Step 1: Import Libraries & Setup GPU

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import timm

import numpy as np
from pathlib import Path
import json
from datetime import datetime
import matplotlib.pyplot as plt
import os

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## Step 2: Load Preprocessed Data (New Best Practice)

In [ ]:
# ============================================================================
# Load data from data_processed_best_practice/ (split first, augmented train only)
# ============================================================================

data_path = Path('data_processed_best_practice')

# Verify directory exists
assert data_path.exists(), f"Data directory not found: {data_path}"

# Load training set (815 images: 163 original + augmented)
X_train = np.load(data_path / 'train' / 'images.npy')
y_train = np.load(data_path / 'train' / 'labels.npy')

# Load validation set (35 ORIGINAL images)
X_val = np.load(data_path / 'val' / 'images.npy')
y_val = np.load(data_path / 'val' / 'labels.npy')

# Load test set (35 ORIGINAL images)
X_test = np.load(data_path / 'test' / 'images.npy')
y_test = np.load(data_path / 'test' / 'labels.npy')

# Load metadata
with open(data_path / 'metadata.json', 'r') as f:
    metadata = json.load(f)

print("✅ Data Loaded Successfully!")
print(f"\nDataset Shape:")
print(f"  Train: {X_train.shape} | Labels: {np.bincount(y_train)}")
print(f"  Val:   {X_val.shape} | Labels: {np.bincount(y_val)}")
print(f"  Test:  {X_test.shape} | Labels: {np.bincount(y_test)}")
print(f"\n📌 Train set has augmented data (815 images)")
print(f"📌 Val & Test sets are ORIGINAL (35 each, no augmentation)")

## Step 3: ImageNet Normalization & Create PyTorch Tensors

In [ ]:
# ImageNet normalization parameters
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406])
IMAGENET_STD = np.array([0.229, 0.224, 0.225])

def normalize_imagenet(images):
    """
    Apply ImageNet normalization
    images: (N, 224, 224, 3) with values in [0, 1]
    returns: (N, 224, 224, 3) normalized
    """
    normalized = np.zeros_like(images)
    for i in range(3):  # RGB channels
        normalized[..., i] = (images[..., i] - IMAGENET_MEAN[i]) / IMAGENET_STD[i]
    return normalized

# Apply normalization
print("Applying ImageNet normalization...")
X_train_norm = normalize_imagenet(X_train)
X_val_norm = normalize_imagenet(X_val)
X_test_norm = normalize_imagenet(X_test)

# Convert to PyTorch tensors and reshape (N, 3, 224, 224) for PyTorch
# PyTorch expects: (Batch, Channels, Height, Width)
X_train_tensor = torch.from_numpy(X_train_norm.transpose(0, 3, 1, 2)).float()
y_train_tensor = torch.from_numpy(y_train).long()

X_val_tensor = torch.from_numpy(X_val_norm.transpose(0, 3, 1, 2)).float()
y_val_tensor = torch.from_numpy(y_val).long()

X_test_tensor = torch.from_numpy(X_test_norm.transpose(0, 3, 1, 2)).float()
y_test_tensor = torch.from_numpy(y_test).long()

# Create PyTorch datasets
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

print("✅ Normalization complete and tensors created")
print(f"   Train tensor: {X_train_tensor.shape} | dtype: {X_train_tensor.dtype}")
print(f"   Val tensor: {X_val_tensor.shape} | dtype: {X_val_tensor.dtype}")
print(f"   Test tensor: {X_test_tensor.shape} | dtype: {X_test_tensor.dtype}")

## Step 4: Create Data Loaders

In [ ]:
# Hyperparameters
BATCH_SIZE = 32
NUM_WORKERS = 0  # Set to 0 on Windows, 4 on Linux/Mac if available

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

print(f"✅ Data loaders created:")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Train batches: {len(train_loader)}")
print(f"   Val batches: {len(val_loader)}")
print(f"   Test batches: {len(test_loader)}")

## Step 5: Build XceptionNet Model (via timm)

In [ ]:
# Load pre-trained XceptionNet from timm
# XceptionNet is available in timm library
model = timm.create_model('xception', pretrained=True, num_classes=2)

# Move to device
model = model.to(device)

# Model info
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"✅ XceptionNet Model Loaded (timm)")
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")
print(f"   Pre-trained: ImageNet")
print(f"   Output classes: 2 (Clear/Turbid)")

## Step 6: Setup Training Configuration

In [ ]:
# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer (Adam with learning rate 0.001)
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=3,
    threshold=1e-4,
    verbose=True
)

# Training parameters
NUM_EPOCHS = 50
PATIENCE = 10  # Early stopping

print(f"✅ Training Configuration:")
print(f"   Loss: CrossEntropyLoss")
print(f"   Optimizer: Adam (lr=0.001)")
print(f"   Scheduler: ReduceLROnPlateau (factor=0.5, patience=3)")
print(f"   Epochs: {NUM_EPOCHS}")
print(f"   Early stopping patience: {PATIENCE}")

## Step 7: Training Loop

In [ ]:
# Training history
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': []
}

best_val_acc = 0
best_val_loss = float('inf')
patience_counter = 0

print("🚀 Starting Training...\n")
start_time = datetime.now()

for epoch in range(NUM_EPOCHS):
    # ========== TRAINING PHASE ==========
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Statistics
        train_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()
    
    train_loss /= len(train_loader)
    train_acc = 100 * train_correct / train_total
    
    # ========== VALIDATION PHASE ==========
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
    
    val_loss /= len(val_loader)
    val_acc = 100 * val_correct / val_total
    
    # Store history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    # Print progress
    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}]")
    print(f"  Train: Loss={train_loss:.4f}, Acc={train_acc:.2f}%")
    print(f"  Val:   Loss={val_loss:.4f}, Acc={val_acc:.2f}%")
    
    # Learning rate scheduling
    scheduler.step(val_loss)
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_val_loss = val_loss
        patience_counter = 0
        
        checkpoint_path = 'xceptionnet_best.pth'
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'val_loss': val_loss
        }, checkpoint_path)
        print(f"  ✅ Best model saved (Val Acc: {val_acc:.2f}%)")
    else:
        patience_counter += 1
        if patience_counter < PATIENCE:
            print(f"  ⏳ No improvement ({patience_counter}/{PATIENCE})")
        else:
            print(f"\n⛔ Early stopping at epoch {epoch+1}")
            break
    
    print()

elapsed_time = datetime.now() - start_time
print(f"✅ Training Complete! Time: {elapsed_time}")

## Step 8: Save Model & Metadata

In [ ]:
# Save final model
final_model_path = "xceptionnet_final_best_practice.pth"
torch.save(model.state_dict(), final_model_path)

# Save metadata
training_metadata = {
    'model': 'XceptionNet',
    'library': 'timm',
    'dataset': 'data_processed_best_practice',
    'dataset_description': 'Split first (70/15/15), augmented training set only',
    'train_images': int(X_train.shape[0]),
    'train_images_original': metadata['train_images_original'],
    'val_images': int(X_val.shape[0]),
    'test_images': int(X_test.shape[0]),
    'best_val_accuracy': float(best_val_acc),
    'best_val_loss': float(best_val_loss),
    'total_epochs': len(history['train_loss']),
    'batch_size': BATCH_SIZE,
    'learning_rate': 0.001,
    'optimizer': 'Adam',
    'scheduler': 'ReduceLROnPlateau',
    'training_date': datetime.now().isoformat(),
    'device': str(device),
    'training_time': str(elapsed_time),
    'training_history': {
        'train_loss': history['train_loss'],
        'train_acc': history['train_acc'],
        'val_loss': history['val_loss'],
        'val_acc': history['val_acc']
    }
}

metadata_path = "xceptionnet_metadata_best_practice.json"
with open(metadata_path, 'w') as f:
    json.dump(training_metadata, f, indent=2)

print(f"✅ Model saved: {final_model_path}")
print(f"✅ Metadata saved: {metadata_path}")
print(f"\n📊 Training Results:")
print(f"   Best Val Accuracy: {best_val_acc:.2f}%")
print(f"   Best Val Loss: {best_val_loss:.4f}")
print(f"   Total Epochs: {len(history['train_loss'])}")
print(f"   Training Time: {elapsed_time}")

## Step 9: Plot Training History

In [ ]:
# Create visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss plot
axes[0].plot(history['train_loss'], label='Train Loss', linewidth=2, marker='o', markersize=3)
axes[0].plot(history['val_loss'], label='Val Loss', linewidth=2, marker='s', markersize=3)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('XceptionNet Training - Loss (Best Practice)', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Accuracy plot
axes[1].plot(history['train_acc'], label='Train Accuracy', linewidth=2, marker='o', markersize=3)
axes[1].plot(history['val_acc'], label='Val Accuracy', linewidth=2, marker='s', markersize=3)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy (%)', fontsize=12)
axes[1].set_title('XceptionNet Training - Accuracy (Best Practice)', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('xceptionnet_training_history_best_practice.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"📊 Training visualization saved: xceptionnet_training_history_best_practice.png")